
# Online Retail: przygotowanie danych do analizy ML w Polars

Pierwsza część notebooka przygotowuje tabelę cech klientów:

> **jeden klient = jeden rekord**

Przepływ:

```text
surowe pozycje zamówień
→ ujednolicenie kolumn i typów
→ kontrola jakości
→ rozdzielenie sprzedaży oraz zwrotów
→ agregacje klienta
→ cechy RFM i dodatkowe wskaźniki
→ zapis tabeli cech do Parquet
```

Notebook obsługuje popularne warianty kolumn zbiorów **Online Retail** i **Online Retail II**, m.in.:

- `InvoiceNo` lub `Invoice`,
- `UnitPrice` lub `Price`,
- `CustomerID` lub `Customer ID`.

## Dane

Umieść plik w katalogu `data/` i ustaw `DATA_PATH` w komórce konfiguracyjnej.

Rekomendowany format do dalszej pracy: **Parquet**.  
Obsługiwane wejścia: CSV i Parquet. W notebooku znajduje się także opcjonalna komórka do jednorazowej konwersji XLSX.


In [ ]:

# W środowisku lokalnym odkomentuj w razie potrzeby:
# %pip install -U polars pyarrow matplotlib fastexcel

from __future__ import annotations

from datetime import timedelta
from pathlib import Path
import re

import polars as pl

print("Polars:", pl.__version__)



## 1. Konfiguracja źródła danych

`scan_csv()` i `scan_parquet()` zwracają `LazyFrame`. Operacje są zapisywane jako plan zapytania, a wykonanie następuje dopiero po wywołaniu `collect()` albo zapisie wyniku.

Jeżeli pliku nie ma, notebook może utworzyć **mały zbiór demonstracyjny**, przeznaczony wyłącznie do sprawdzenia kodu. Do właściwego wykładu należy użyć pełnego zbioru Online Retail.


In [ ]:

DATA_PATH = Path("data/online_retail.parquet")

# Mały fallback pozwala uruchomić notebook bez pobranego zbioru.
# Nie należy interpretować wyników biznesowo.
CREATE_DEMO_IF_MISSING = True

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:

def create_demo_dataset(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)

    demo = pl.DataFrame(
        {
            "InvoiceNo": [
                "10001", "10001", "10002", "10003", "C10003",
                "10004", "10005", "10006", "10006", "10007",
                "10008", "10009", "10010", "10011"
            ],
            "StockCode": [
                "A1", "B1", "A1", "C1", "C1",
                "D1", "A1", "B1", "E1", "F1",
                "A1", "G1", "H1", "B1"
            ],
            "Description": [
                "WHITE HANGING HEART", "RED RETROSPOT BAG", "WHITE HANGING HEART",
                "CERAMIC MUG", "CERAMIC MUG", "WOODEN FRAME",
                "WHITE HANGING HEART", "RED RETROSPOT BAG", "METAL LANTERN",
                "PAPER NOTEBOOK", "WHITE HANGING HEART", "VINTAGE CLOCK",
                "GLASS CANDLE HOLDER", "RED RETROSPOT BAG"
            ],
            "Quantity": [2, 1, 3, 4, -1, 1, 2, 2, 1, 5, 1, 1, 2, 3],
            "InvoiceDate": [
                "2011-01-03 10:00", "2011-01-03 10:00", "2011-01-20 12:00",
                "2011-02-01 09:30", "2011-02-05 11:00", "2011-02-10 16:00",
                "2011-03-02 14:00", "2011-03-15 10:00", "2011-03-15 10:00",
                "2011-03-20 13:00", "2011-04-01 08:00", "2011-04-12 17:00",
                "2011-05-02 10:30", "2011-05-18 15:00"
            ],
            "UnitPrice": [2.5, 4.0, 2.5, 6.0, 6.0, 8.0, 2.5, 4.0, 10.0, 3.0, 2.5, 15.0, 7.0, 4.0],
            "CustomerID": [101, 101, 101, 102, 102, 102, 103, 103, 103, 104, 104, 105, None, 103],
            "Country": ["United Kingdom"] * 14,
        }
    )
    demo.write_parquet(path)


if not DATA_PATH.exists():
    if not CREATE_DEMO_IF_MISSING:
        raise FileNotFoundError(
            f"Nie znaleziono {DATA_PATH}. Ustaw poprawną ścieżkę do CSV lub Parquet."
        )

    create_demo_dataset(DATA_PATH)
    print(
        f"UWAGA: utworzono mały zbiór demonstracyjny: {DATA_PATH}. "
        "Do prezentacji użyj pełnego zbioru Online Retail."
    )



### Opcjonalnie: jednorazowa konwersja XLSX do Parquet

Oficjalne warianty zbioru bywają udostępniane jako XLSX. Excel nie jest formatem odpowiednim do leniwego skanowania, dlatego warto przekonwertować go raz do Parquet.

Dla pliku Online Retail II zawierającego kilka arkuszy należy wczytać arkusze osobno, połączyć je i zapisać jako jeden plik.


In [ ]:

# Uruchom tylko wtedy, gdy masz plik XLSX.
#
# XLSX_PATH = Path("data/online_retail.xlsx")
# df_xlsx = pl.read_excel(XLSX_PATH)
# df_xlsx.write_parquet("data/online_retail.parquet")
#
# Po konwersji ustaw:
# DATA_PATH = Path("data/online_retail.parquet")



## 2. Leniwe wczytanie danych

Nie wywołujemy jeszcze `collect()`. Na tym etapie budujemy jedynie początek planu przetwarzania.


In [ ]:

def scan_dataset(path: Path) -> pl.LazyFrame:
    suffix = path.suffix.lower()

    if suffix == ".parquet":
        return pl.scan_parquet(path)

    if suffix in {".csv", ".txt"}:
        return pl.scan_csv(
            path,
            try_parse_dates=False,
            infer_schema_length=10_000,
            ignore_errors=False,
        )

    raise ValueError(
        f"Nieobsługiwany format {suffix!r}. Użyj CSV albo Parquet."
    )


raw_lf = scan_dataset(DATA_PATH)
raw_lf.collect_schema()



## 3. Ujednolicenie nazw kolumn

Różne wersje zbioru używają nieco innych nazw. Funkcja poniżej mapuje je na jeden spójny schemat.


In [ ]:

COLUMN_ALIASES: dict[str, tuple[str, ...]] = {
    "invoice_no": ("invoiceno", "invoice"),
    "stock_code": ("stockcode",),
    "description": ("description",),
    "quantity": ("quantity",),
    "invoice_date": ("invoicedate",),
    "unit_price": ("unitprice", "price"),
    "customer_id": ("customerid",),
    "country": ("country",),
}


def normalized_name(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", name.lower())


def canonicalize_columns(lf: pl.LazyFrame) -> pl.LazyFrame:
    source_columns = lf.collect_schema().names()
    lookup = {normalized_name(column): column for column in source_columns}

    rename_map: dict[str, str] = {}

    for target, aliases in COLUMN_ALIASES.items():
        for alias in aliases:
            if alias in lookup:
                rename_map[lookup[alias]] = target
                break

    required = {
        "invoice_no",
        "stock_code",
        "quantity",
        "invoice_date",
        "unit_price",
        "customer_id",
    }
    available_targets = set(rename_map.values())
    missing = required - available_targets

    if missing:
        raise ValueError(
            "Brakuje wymaganych kolumn po normalizacji: "
            f"{sorted(missing)}. Kolumny źródłowe: {source_columns}"
        )

    lf = lf.rename(rename_map)

    # Niektóre wersje danych mogą nie zawierać opisu lub kraju.
    current = set(lf.collect_schema().names())
    optional_columns = []
    if "description" not in current:
        optional_columns.append(pl.lit(None, dtype=pl.String).alias("description"))
    if "country" not in current:
        optional_columns.append(pl.lit(None, dtype=pl.String).alias("country"))

    return lf.with_columns(optional_columns) if optional_columns else lf


canonical_lf = canonicalize_columns(raw_lf)
canonical_lf.collect_schema()



## 4. Typy danych i flagi jakości

Przed filtrowaniem tworzymy flagi:

- `is_cancelled` — numer faktury zaczyna się od `C`,
- `is_return` — anulowanie albo ujemna ilość,
- `line_revenue` — wartość pozycji.

Informację o zwrotach zachowujemy **przed usunięciem anulowanych pozycji**, ponieważ później wykorzystamy ją do obliczenia `return_rate`.


In [ ]:

typed_lf = (
    canonical_lf
    .with_columns(
        pl.col("invoice_no").cast(pl.String, strict=False).str.strip_chars(),
        pl.col("stock_code").cast(pl.String, strict=False).str.strip_chars(),
        pl.col("description").cast(pl.String, strict=False).str.strip_chars(),
        pl.col("quantity").cast(pl.Float64, strict=False),
        pl.col("unit_price").cast(pl.Float64, strict=False),
        pl.col("invoice_date")
        .cast(pl.String, strict=False)
        .str.to_datetime(strict=False),
        pl.col("customer_id")
        .cast(pl.Float64, strict=False)
        .cast(pl.Int64, strict=False)
        .cast(pl.String, strict=False),
        pl.col("country").cast(pl.String, strict=False).str.strip_chars(),
    )
    .with_columns(
        pl.col("invoice_no")
        .str.to_uppercase()
        .str.starts_with("C")
        .fill_null(False)
        .alias("is_cancelled"),
    )
    .with_columns(
        (
            pl.col("is_cancelled")
            | (pl.col("quantity").fill_null(0) < 0)
        ).alias("is_return"),
        (
            pl.col("quantity") * pl.col("unit_price")
        ).alias("line_revenue"),
    )
)

typed_lf.select(
    "invoice_no",
    "customer_id",
    "invoice_date",
    "quantity",
    "unit_price",
    "is_cancelled",
    "is_return",
    "line_revenue",
).head(10).collect()



## 5. Kontrola jakości danych

Zanim usuniemy rekordy, sprawdzamy skalę problemów. W prawdziwej analizie warto zapisać taki raport jako artefakt procesu.


In [ ]:

quality_report = typed_lf.select(
    pl.len().alias("rows_total"),
    pl.col("customer_id").is_null().sum().alias("missing_customer_id"),
    pl.col("invoice_date").is_null().sum().alias("invalid_or_missing_date"),
    pl.col("quantity").is_null().sum().alias("missing_quantity"),
    pl.col("unit_price").is_null().sum().alias("missing_unit_price"),
    (pl.col("quantity") <= 0).sum().alias("non_positive_quantity"),
    (pl.col("unit_price") <= 0).sum().alias("non_positive_price"),
    pl.col("is_cancelled").sum().alias("cancelled_rows"),
    pl.col("is_return").sum().alias("return_rows"),
).collect()

quality_report



## 6. Rozdzielenie poprawnej sprzedaży i informacji o zwrotach

Do profilu zakupowego wykorzystujemy wyłącznie poprawne, dodatnie transakcje sprzedażowe:

- klient jest znany,
- data jest poprawna,
- faktura nie jest anulowana,
- ilość jest dodatnia,
- cena jest dodatnia.

Zwroty agregujemy osobno na podstawie danych sprzed filtrowania.


In [ ]:

valid_sales_lf = typed_lf.filter(
    pl.col("customer_id").is_not_null()
    & pl.col("invoice_date").is_not_null()
    & ~pl.col("is_cancelled")
    & (pl.col("quantity") > 0)
    & (pl.col("unit_price") > 0)
)

return_stats_lf = (
    typed_lf
    .filter(pl.col("customer_id").is_not_null())
    .group_by("customer_id")
    .agg(
        pl.len().alias("all_line_count"),
        pl.col("is_return").sum().alias("return_line_count"),
        pl.when(pl.col("is_return"))
        .then(pl.col("line_revenue").abs())
        .otherwise(0.0)
        .sum()
        .alias("absolute_return_value"),
    )
)

cleaning_summary = pl.concat(
    [
        typed_lf.select(
            pl.lit("raw").alias("stage"),
            pl.len().alias("rows"),
            pl.col("customer_id").n_unique().alias("customers"),
        ),
        valid_sales_lf.select(
            pl.lit("valid_sales").alias("stage"),
            pl.len().alias("rows"),
            pl.col("customer_id").n_unique().alias("customers"),
        ),
    ]
).collect()

cleaning_summary



## 7. Data referencyjna i cechy RFM

RFM:

- **Recency** — ile dni upłynęło od ostatniego zakupu,
- **Frequency** — ile unikalnych zamówień złożył klient,
- **Monetary** — jaka była łączna wartość zakupów.

Datę referencyjną ustawiamy na dzień po ostatniej poprawnej transakcji w zbiorze. Dzięki temu `recency_days` jest dodatnią i łatwą do interpretacji liczbą.


In [ ]:

max_invoice_date = (
    valid_sales_lf
    .select(pl.col("invoice_date").max())
    .collect()
    .item()
)

if max_invoice_date is None:
    raise ValueError("Po czyszczeniu nie pozostały żadne poprawne transakcje.")

snapshot_date = max_invoice_date + timedelta(days=1)
snapshot_date



## 8. Agregacja: jeden klient = jeden rekord

Poza RFM tworzymy także:

- średnią wartość zamówienia,
- liczbę unikalnych produktów,
- łączną liczbę sztuk,
- długość aktywności klienta,
- orientacyjną częstotliwość zamówień w przeliczeniu na 30 dni,
- udział pozycji zwrotów/anulowań.


In [ ]:

customer_features_lf = (
    valid_sales_lf
    .group_by("customer_id")
    .agg(
        pl.col("invoice_date").min().alias("first_purchase_date"),
        pl.col("invoice_date").max().alias("last_purchase_date"),
        pl.col("invoice_no").n_unique().alias("number_of_orders"),
        pl.col("line_revenue").sum().alias("total_revenue"),
        pl.col("stock_code").n_unique().alias("number_of_products"),
        pl.col("quantity").sum().alias("total_items"),
        pl.col("country").mode().first().alias("main_country"),
    )
    .with_columns(
        (
            pl.lit(snapshot_date) - pl.col("last_purchase_date")
        ).dt.total_days().alias("recency_days"),
        (
            (
                pl.col("last_purchase_date")
                - pl.col("first_purchase_date")
            ).dt.total_days()
            + 1
        ).alias("active_days"),
    )
    .with_columns(
        (
            pl.col("total_revenue")
            / pl.col("number_of_orders")
        ).alias("average_order_value"),
        (
            pl.col("number_of_orders")
            / pl.max_horizontal(pl.col("active_days"), pl.lit(1))
            * 30
        ).alias("purchase_frequency_30d"),
    )
    .join(return_stats_lf, on="customer_id", how="left")
    .with_columns(
        pl.col("return_line_count").fill_null(0),
        pl.col("all_line_count").fill_null(0),
        pl.col("absolute_return_value").fill_null(0.0),
    )
    .with_columns(
        pl.when(pl.col("all_line_count") > 0)
        .then(
            pl.col("return_line_count")
            / pl.col("all_line_count")
        )
        .otherwise(0.0)
        .alias("return_rate")
    )
    .select(
        "customer_id",
        "recency_days",
        "number_of_orders",
        "total_revenue",
        "average_order_value",
        "number_of_products",
        "purchase_frequency_30d",
        "return_rate",
        "total_items",
        "active_days",
        "first_purchase_date",
        "last_purchase_date",
        "main_country",
    )
    .sort("customer_id")
)

customer_features = customer_features_lf.collect()
customer_features



## 9. Sprawdzenie gotowej tabeli cech

Przed przekazaniem danych do modelu sprawdzamy:

- czy jeden klient występuje tylko raz,
- czy nie ma braków w najważniejszych cechach,
- czy wartości są w sensownych zakresach.


In [ ]:

feature_check = customer_features.select(
    pl.len().alias("rows"),
    pl.col("customer_id").n_unique().alias("unique_customers"),
    pl.any_horizontal(
        pl.col(
            "recency_days",
            "number_of_orders",
            "total_revenue",
            "average_order_value",
            "number_of_products",
            "purchase_frequency_30d",
            "return_rate",
        ).is_null()
    ).sum().alias("rows_with_missing_model_features"),
    pl.col("total_revenue").min().alias("min_total_revenue"),
    pl.col("total_revenue").max().alias("max_total_revenue"),
    pl.col("return_rate").min().alias("min_return_rate"),
    pl.col("return_rate").max().alias("max_return_rate"),
)

feature_check


In [ ]:

assert customer_features.height == customer_features["customer_id"].n_unique()
assert customer_features["return_rate"].min() >= 0
assert customer_features["return_rate"].max() <= 1
assert customer_features["number_of_orders"].min() >= 1

print("Walidacja tabeli cech zakończona poprawnie.")



## 10. Podstawowe statystyki cech

Na tym etapie nie budujemy jeszcze modelu. Sprawdzamy jedynie, czy tabela cech ma sens analityczny i czy rozkłady nie wskazują oczywistych problemów.


In [ ]:

MODEL_FEATURES = [
    "recency_days",
    "number_of_orders",
    "total_revenue",
    "average_order_value",
    "number_of_products",
    "purchase_frequency_30d",
    "return_rate",
]

customer_features.select(MODEL_FEATURES).describe()



## 11. Zapis tabeli cech

Parquet zachowuje typy danych i dobrze nadaje się do kolejnych części notebooka:

- skalowanie,
- K-Means,
- DBSCAN,
- PCA,
- wykrywanie anomalii.


In [ ]:

FEATURES_PATH = OUTPUT_DIR / "online_retail_customer_features.parquet"
customer_features.write_parquet(FEATURES_PATH)

print(f"Zapisano {customer_features.height:,} klientów do: {FEATURES_PATH}")



## Podsumowanie części 1

Przekształciliśmy dane transakcyjne do postaci odpowiedniej dla algorytmów ML:

```text
wiele pozycji zamówień jednego klienta
→ jeden rekord opisujący zachowanie klienta
```

Najważniejsza lekcja:

> Algorytm ML nie powinien otrzymywać przypadkowej surowej tabeli.  
> Analityk musi najpierw zdefiniować jednostkę analizy i przygotować sensowne cechy.

W następnej części wykorzystamy utworzoną tabelę do segmentacji klientów za pomocą K-Means i DBSCAN.



---

## Część 2. Grupowanie danych i odkrywanie segmentów

Cel:

> znaleźć grupy klientów o podobnych zachowaniach, mimo że w danych nie istnieje gotowa etykieta segmentu.

Przepływ:

```text
tabela cech klientów
→ transformacja skośnych zmiennych
→ StandardScaler
→ wybór liczby klastrów
→ K-Means
→ profile segmentów
→ interpretacja biznesowa
```

Na końcu porównamy K-Means z DBSCAN, który może pozostawić część klientów poza klastrami jako `noise`.



### 12. Przygotowanie środowiska i tabeli klientów

Jeżeli poprzednia część notebooka została już wykonana, korzystamy z obiektu `customer_features`.  
W przeciwnym razie próbujemy wczytać zapisany plik Parquet.


In [ ]:

# W środowisku lokalnym odkomentuj w razie potrzeby:
# %pip install -U scikit-learn matplotlib

from __future__ import annotations

import math
import numpy as np
import matplotlib.pyplot as plt

from sklearn.cluster import DBSCAN, KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

try:
    customer_features
except NameError:
    features_path = Path("outputs/online_retail_customer_features.parquet")
    if not features_path.exists():
        raise FileNotFoundError(
            "Nie znaleziono tabeli cech. Najpierw wykonaj część 1 notebooka."
        )
    customer_features = pl.read_parquet(features_path)

print(f"Liczba klientów: {customer_features.height:,}")
customer_features.head()



### 13. Wybór cech do segmentacji

Do przykładu wybieramy cechy opisujące:

- aktualność relacji z klientem,
- liczbę i wartość zamówień,
- różnorodność kupowanych produktów,
- częstotliwość zakupów,
- udział zwrotów.

`customer_id` nie jest cechą modelu — jest wyłącznie identyfikatorem.


In [ ]:

CLUSTER_FEATURES = [
    "recency_days",
    "number_of_orders",
    "total_revenue",
    "average_order_value",
    "number_of_products",
    "purchase_frequency_30d",
    "return_rate",
]

missing_features = [
    column for column in CLUSTER_FEATURES
    if column not in customer_features.columns
]
if missing_features:
    raise ValueError(f"Brakuje cech: {missing_features}")

model_frame = customer_features.select(["customer_id", *CLUSTER_FEATURES])

null_counts = model_frame.select(
    [pl.col(column).is_null().sum().alias(column) for column in CLUSTER_FEATURES]
)
null_counts



### 14. Transformacja cech o silnie skośnych rozkładach

W danych retailowych niewielka liczba klientów może składać bardzo dużo zamówień i generować bardzo wysoki przychód.

K-Means minimalizuje odległości kwadratowe, dlatego ekstremalne wartości mogą mocno przesuwać centroidy. Dla nieujemnych cech zakupowych stosujemy `log1p`, czyli:

```text
log(1 + x)
```

Transformacja zachowuje kolejność wartości, ale zmniejsza różnice między klientami typowymi i ekstremalnymi.

Nie transformujemy w ten sposób `return_rate`, ponieważ jest już udziałem z przedziału od 0 do 1.


In [ ]:

LOG1P_FEATURES = [
    "recency_days",
    "number_of_orders",
    "total_revenue",
    "average_order_value",
    "number_of_products",
    "purchase_frequency_30d",
]

transformed_frame = model_frame.with_columns(
    [
        pl.col(column)
        .clip(lower_bound=0)
        .log1p()
        .alias(column)
        for column in LOG1P_FEATURES
    ]
)

transformed_frame.select(CLUSTER_FEATURES).describe()



### 15. Dlaczego skalujemy dane?

Przykładowe zakresy mogą być bardzo różne:

- `return_rate`: od 0 do 1,
- `number_of_orders`: od kilku do setek,
- `total_revenue`: od małych kwot do dziesiątek tysięcy.

Bez skalowania zmienne o dużych wartościach liczbowych zdominowałyby odległość euklidesową.

`StandardScaler` dla każdej cechy odejmuje średnią i dzieli wynik przez odchylenie standardowe. Po transformacji cechy mają porównywalną skalę.


In [ ]:

X = transformed_frame.select(CLUSTER_FEATURES).to_numpy()

if not np.isfinite(X).all():
    raise ValueError("Macierz cech zawiera NaN lub wartości nieskończone.")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

scaling_check = pl.DataFrame(
    {
        "feature": CLUSTER_FEATURES,
        "mean_after_scaling": X_scaled.mean(axis=0),
        "std_after_scaling": X_scaled.std(axis=0),
    }
)
scaling_check



### 16. Czym jest centroid?

W K-Means każdy klaster jest reprezentowany przez **centroid**, czyli średni punkt wszystkich obserwacji należących do danej grupy.

Algorytm iteracyjnie:

1. inicjalizuje centroidy,
2. przypisuje każdy rekord do najbliższego centroidu,
3. przelicza położenie centroidów,
4. powtarza kroki do ustabilizowania rozwiązania.

Musimy wcześniej określić liczbę klastrów `k`.



### 17. Wybór liczby klastrów

Sprawdzimy kilka wartości `k` za pomocą dwóch wskaźników:

- **inertia** — suma kwadratów odległości rekordów od ich centroidów; zawsze maleje wraz ze wzrostem `k`,
- **silhouette score** — porównuje spójność rekordu z własnym klastrem i jego oddzielenie od najbliższego innego klastra.

Metoda łokcia szuka miejsca, po którym dalszy spadek inertia staje się znacznie mniejszy.  
Silhouette score jest pomocą, ale nie zastępuje oceny biznesowej użyteczności segmentów.


In [ ]:

n_customers = X_scaled.shape[0]
if n_customers < 3:
    raise ValueError("Do oceny klastrów potrzebujemy co najmniej 3 klientów.")

max_k = min(8, n_customers - 1)
candidate_k = list(range(2, max_k + 1))

k_results: list[dict[str, float | int]] = []

for k in candidate_k:
    model = KMeans(
        n_clusters=k,
        init="k-means++",
        n_init=20,
        random_state=42,
    )
    labels = model.fit_predict(X_scaled)

    score = silhouette_score(
        X_scaled,
        labels,
        sample_size=min(5_000, n_customers),
        random_state=42,
    )

    k_results.append(
        {
            "k": k,
            "inertia": float(model.inertia_),
            "silhouette_score": float(score),
        }
    )

k_evaluation = pl.DataFrame(k_results)
k_evaluation


In [ ]:

plt.figure(figsize=(8, 4.5))
plt.plot(
    k_evaluation["k"].to_list(),
    k_evaluation["inertia"].to_list(),
    marker="o",
)
plt.title("Metoda łokcia")
plt.xlabel("Liczba klastrów k")
plt.ylabel("Inertia")
plt.xticks(k_evaluation["k"].to_list())
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:

plt.figure(figsize=(8, 4.5))
plt.plot(
    k_evaluation["k"].to_list(),
    k_evaluation["silhouette_score"].to_list(),
    marker="o",
)
plt.title("Silhouette score dla różnych wartości k")
plt.xlabel("Liczba klastrów k")
plt.ylabel("Silhouette score")
plt.xticks(k_evaluation["k"].to_list())
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()



### 18. Wybór `k` i finalny model K-Means

Notebook podpowiada wartość z najwyższym silhouette score. Przed prezentacją warto jednak ustawić `SELECTED_K` świadomie po obejrzeniu obu wykresów i profili biznesowych.

Większa liczba klastrów nie musi oznaczać lepszej segmentacji. Segmenty powinny być:

- wystarczająco różne,
- odpowiednio liczne,
- możliwe do opisania,
- przydatne do podjęcia działań.


In [ ]:

recommended_k = int(
    k_evaluation
    .sort("silhouette_score", descending=True)
    .row(0, named=True)["k"]
)

# Możesz zmienić tę wartość po analizie wykresów.
SELECTED_K = recommended_k

print(f"Najwyższy silhouette score uzyskano dla k={recommended_k}.")
print(f"Finalny model zostanie uruchomiony dla k={SELECTED_K}.")


In [ ]:

kmeans = KMeans(
    n_clusters=SELECTED_K,
    init="k-means++",
    n_init=20,
    random_state=42,
)

kmeans_labels = kmeans.fit_predict(X_scaled)

customer_segments = customer_features.with_columns(
    pl.Series("cluster_id", kmeans_labels)
)

customer_segments.select(
    "customer_id",
    "cluster_id",
    *CLUSTER_FEATURES,
).head(10)



### 19. Profile klastrów

Sam numer klastra niczego jeszcze nie wyjaśnia.

Dla każdego klastra obliczamy:

- liczbę i udział klientów,
- mediany najważniejszych cech.

Mediana jest tu przydatna, ponieważ profile klientów nadal mogą zawierać wartości ekstremalne.


In [ ]:

profile_aggregations = [
    pl.len().alias("customers"),
    *[
        pl.col(column).median().round(3).alias(f"median_{column}")
        for column in CLUSTER_FEATURES
    ],
]

cluster_profiles = (
    customer_segments
    .group_by("cluster_id")
    .agg(profile_aggregations)
    .with_columns(
        (
            pl.col("customers") / customer_segments.height
        ).round(4).alias("customer_share")
    )
    .sort("cluster_id")
)

cluster_profiles



### 20. Profil względny

Poniższa tabela pokazuje średnie wartości **po skalowaniu**:

- wartość dodatnia — klaster znajduje się powyżej średniej zbioru,
- wartość ujemna — klaster znajduje się poniżej średniej,
- wartość bliska zeru — klaster jest zbliżony do przeciętnego klienta.

Dzięki temu łatwiej porównywać cechy o różnych jednostkach.


In [ ]:

scaled_frame = pl.DataFrame(
    X_scaled,
    schema=CLUSTER_FEATURES,
    orient="row",
).with_columns(
    pl.Series("cluster_id", kmeans_labels)
)

relative_profiles = (
    scaled_frame
    .group_by("cluster_id")
    .agg(
        [
            pl.col(column).mean().round(3).alias(column)
            for column in CLUSTER_FEATURES
        ]
    )
    .sort("cluster_id")
)

relative_profiles


In [ ]:

profile_matrix = relative_profiles.select(CLUSTER_FEATURES).to_numpy()
cluster_labels_for_plot = relative_profiles["cluster_id"].to_list()

plt.figure(figsize=(10, max(3.5, 0.8 * len(cluster_labels_for_plot))))
image = plt.imshow(profile_matrix, aspect="auto")
plt.colorbar(image, label="Średnia standaryzowana")
plt.xticks(
    range(len(CLUSTER_FEATURES)),
    CLUSTER_FEATURES,
    rotation=45,
    ha="right",
)
plt.yticks(
    range(len(cluster_labels_for_plot)),
    [f"cluster_{cluster_id}" for cluster_id in cluster_labels_for_plot],
)
plt.title("Względna charakterystyka klastrów")
plt.tight_layout()
plt.show()



### 21. Nadawanie nazw segmentom

K-Means zwraca techniczne identyfikatory:

```text
cluster_0
cluster_1
cluster_2
```

Numery nie mają stałego znaczenia i przy innym uczeniu mogą zostać zamienione miejscami.

Analityk powinien na podstawie profili nadać segmentom nazwy biznesowe, np.:

- klienci okazjonalni,
- klienci regularni,
- klienci wysokowartościowi,
- klienci traceni,
- klienci hurtowi.

Nazwy poniżej należy uzupełnić **dopiero po analizie otrzymanych profili**.


In [ ]:

# Przykład po ręcznej interpretacji wyników:
#
# CLUSTER_NAMES = {
#     0: "Klienci okazjonalni",
#     1: "Klienci wysokowartościowi",
#     2: "Klienci traceni",
#     3: "Klienci regularni",
# }

CLUSTER_NAMES = {
    cluster_id: f"Segment {cluster_id} — do interpretacji"
    for cluster_id in customer_segments["cluster_id"].unique().sort().to_list()
}

customer_segments = customer_segments.with_columns(
    pl.col("cluster_id")
    .replace_strict(CLUSTER_NAMES)
    .alias("segment_name")
)

customer_segments.select(
    "customer_id",
    "cluster_id",
    "segment_name",
    "recency_days",
    "number_of_orders",
    "total_revenue",
).head(10)



## 22. Porównanie z DBSCAN

DBSCAN:

- nie wymaga wcześniejszego podania liczby klastrów,
- buduje grupy na podstawie lokalnej gęstości,
- może oznaczyć część rekordów jako `-1`, czyli szum,
- jest wrażliwy na dobór `eps`, `min_samples`, skalowanie i liczbę wymiarów.

W przeciwieństwie do K-Means nie gwarantuje, że każdy klient otrzyma zwykły segment.



### 23. Krótkie strojenie parametru `eps`

`eps` określa promień sąsiedztwa w przestrzeni przeskalowanych cech.  
Dla kilku wartości sprawdzamy:

- liczbę znalezionych klastrów,
- udział rekordów oznaczonych jako szum,
- silhouette score liczony tylko dla rekordów przypisanych do zwykłych klastrów.


In [ ]:

DBSCAN_MIN_SAMPLES = 10 if n_customers >= 100 else 3
EPS_CANDIDATES = [0.4, 0.6, 0.8, 1.0, 1.2, 1.5, 2.0]

dbscan_results: list[dict[str, float | int | None]] = []

for eps in EPS_CANDIDATES:
    labels = DBSCAN(
        eps=eps,
        min_samples=DBSCAN_MIN_SAMPLES,
    ).fit_predict(X_scaled)

    cluster_ids = set(labels)
    n_clusters = len(cluster_ids - {-1})
    noise_share = float(np.mean(labels == -1))

    non_noise_mask = labels != -1
    non_noise_labels = labels[non_noise_mask]
    non_noise_clusters = set(non_noise_labels)

    score: float | None = None
    if (
        len(non_noise_clusters) >= 2
        and non_noise_mask.sum() > len(non_noise_clusters)
    ):
        score = float(
            silhouette_score(
                X_scaled[non_noise_mask],
                non_noise_labels,
                sample_size=min(5_000, int(non_noise_mask.sum())),
                random_state=42,
            )
        )

    dbscan_results.append(
        {
            "eps": eps,
            "min_samples": DBSCAN_MIN_SAMPLES,
            "clusters": n_clusters,
            "noise_share": noise_share,
            "silhouette_without_noise": score,
        }
    )

dbscan_evaluation = pl.DataFrame(dbscan_results)
dbscan_evaluation



### 24. Finalny DBSCAN

Automatyczna podpowiedź wybiera spośród rozsądnych kandydatów konfigurację z najwyższym silhouette score. Jest to tylko punkt startowy — interpretacja liczby grup i udziału szumu nadal należy do analityka.


In [ ]:

valid_dbscan_candidates = dbscan_evaluation.filter(
    (pl.col("clusters") >= 2)
    & (pl.col("clusters") <= 10)
    & (pl.col("noise_share") <= 0.60)
    & pl.col("silhouette_without_noise").is_not_null()
)

if valid_dbscan_candidates.height > 0:
    DBSCAN_EPS = float(
        valid_dbscan_candidates
        .sort("silhouette_without_noise", descending=True)
        .row(0, named=True)["eps"]
    )
else:
    DBSCAN_EPS = 1.0

print(
    f"Wybrane parametry: eps={DBSCAN_EPS}, "
    f"min_samples={DBSCAN_MIN_SAMPLES}"
)


In [ ]:

dbscan = DBSCAN(
    eps=DBSCAN_EPS,
    min_samples=DBSCAN_MIN_SAMPLES,
)

dbscan_labels = dbscan.fit_predict(X_scaled)

customer_segments = customer_segments.with_columns(
    pl.Series("dbscan_cluster_id", dbscan_labels),
    pl.Series("dbscan_is_noise", dbscan_labels == -1),
)

dbscan_summary = (
    customer_segments
    .group_by("dbscan_cluster_id")
    .agg(
        pl.len().alias("customers"),
        pl.col("total_revenue").median().round(2).alias("median_total_revenue"),
        pl.col("number_of_orders").median().round(2).alias("median_orders"),
        pl.col("recency_days").median().round(2).alias("median_recency_days"),
    )
    .with_columns(
        (
            pl.col("customers") / customer_segments.height
        ).round(4).alias("customer_share")
    )
    .sort("dbscan_cluster_id")
)

dbscan_summary


In [ ]:

comparison = pl.DataFrame(
    {
        "method": ["K-Means", "DBSCAN"],
        "regular_clusters": [
            customer_segments["cluster_id"].n_unique(),
            len(set(dbscan_labels) - {-1}),
        ],
        "customers_marked_as_noise": [
            0,
            int(np.sum(dbscan_labels == -1)),
        ],
        "noise_share": [
            0.0,
            float(np.mean(dbscan_labels == -1)),
        ],
    }
)

comparison



## 25. Interpretacja porównania

### K-Means

Najczęściej lepiej pasuje do klasycznej segmentacji biznesowej, gdy:

- chcemy przypisać każdego klienta do jednej grupy,
- potrzebujemy stabilnej i łatwej do komunikacji liczby segmentów,
- grupy mają być później wykorzystywane w raportach lub kampaniach.

### DBSCAN

Jest interesujący, gdy:

- spodziewamy się nieregularnych skupisk,
- nie znamy liczby grup,
- rekordy niepasujące do żadnego gęstego obszaru są ważnym wynikiem,
- chcemy znaleźć lokalne skupiska albo potencjalne anomalie.

W danych klientów o kilku cechach DBSCAN może oznaczyć wielu klientów jako szum. Nie musi to oznaczać błędu algorytmu — może wskazywać, że przy danej definicji odległości dane nie tworzą wyraźnych obszarów o podobnej gęstości.



## Podsumowanie części 2

Przeprowadziliśmy pełny proces segmentacji:

```text
cechy klientów
→ log1p dla skośnych zmiennych
→ StandardScaler
→ metoda łokcia i silhouette score
→ K-Means
→ profile klastrów
→ nazwy biznesowe
```

Porównaliśmy go z DBSCAN:

```text
cechy klientów
→ DBSCAN
→ klastry gęstościowe
→ rekordy oznaczone jako noise
```

Najważniejsza lekcja:

> Algorytm znajduje matematyczne grupy, ale to analityk ocenia, czy grupy są stabilne, zrozumiałe i przydatne biznesowo.



## 26. Zapis wyników segmentacji

Zapisujemy tabelę klientów wraz z identyfikatorami K-Means i DBSCAN.  
Plik może zostać później wykorzystany w PCA, analizie anomalii oraz aplikacji Streamlit.


In [ ]:

SEGMENTS_PATH = OUTPUT_DIR / "online_retail_customer_segments.parquet"
customer_segments.write_parquet(SEGMENTS_PATH)

print(
    f"Zapisano segmentację {customer_segments.height:,} klientów do: "
    f"{SEGMENTS_PATH}"
)



---

## Część 3. Redukcja wymiarów za pomocą PCA

Cel:

> zredukować wielowymiarowy profil klienta do dwóch komponentów i przedstawić segmenty K-Means na wykresie.

Przepływ:

```text
cechy klientów
→ transformacja log1p
→ StandardScaler
→ PCA(n_components=2)
→ wykres segmentów
```

PCA nie wybiera dwóch istniejących kolumn. Tworzy dwie nowe zmienne będące kombinacjami cech wejściowych.



### 27. Przygotowanie danych do PCA

Korzystamy dokładnie z tej samej przeskalowanej macierzy `X_scaled`, na której wcześniej uruchomiliśmy K-Means. Dzięki temu wizualizacja pokazuje tę samą reprezentację klientów, która była podstawą segmentacji.


In [ ]:

from sklearn.decomposition import PCA

try:
    X_scaled
    customer_segments
except NameError as exc:
    raise RuntimeError(
        "Najpierw wykonaj część 2 notebooka z przygotowaniem X_scaled "
        "i segmentacją K-Means."
    ) from exc

pca_2d = PCA(n_components=2)
X_pca_2d = pca_2d.fit_transform(X_scaled)

print("Kształt przed PCA:", X_scaled.shape)
print("Kształt po PCA:", X_pca_2d.shape)



### 28. Explained variance

`explained_variance_ratio_` pokazuje, jaka część zmienności danych jest reprezentowana przez każdy komponent.

Jeżeli dwa komponenty wyjaśniają np. 65% wariancji, oznacza to, że wykres 2D zachowuje dużą, ale niepełną część informacji zawartej w oryginalnych cechach.


In [ ]:

explained_variance = pl.DataFrame(
    {
        "component": ["PC1", "PC2"],
        "explained_variance_ratio": pca_2d.explained_variance_ratio_,
        "cumulative_explained_variance": np.cumsum(
            pca_2d.explained_variance_ratio_
        ),
    }
)

explained_variance


In [ ]:

plt.figure(figsize=(7, 4.5))
plt.bar(
    explained_variance["component"].to_list(),
    explained_variance["explained_variance_ratio"].to_list(),
)
plt.title("Udział wariancji wyjaśnionej przez komponenty PCA")
plt.xlabel("Komponent")
plt.ylabel("Explained variance ratio")
plt.tight_layout()
plt.show()



### 29. Wizualizacja segmentów klientów

Punkty blisko siebie mają podobne profile w wielowymiarowej przestrzeni cech. Kolor grupy nie jest tu ustawiany ręcznie — wykres używa domyślnej mapy Matplotlib na podstawie `cluster_id`.

Pamiętajmy, że jest to projekcja. Nakładanie się punktów na wykresie 2D nie musi oznaczać, że klastry są identyczne w pełnej przestrzeni.


In [ ]:

pca_plot_frame = customer_segments.select(
    "customer_id",
    "cluster_id",
).with_columns(
    pl.Series("PC1", X_pca_2d[:, 0]),
    pl.Series("PC2", X_pca_2d[:, 1]),
)

plt.figure(figsize=(8.5, 6))
scatter = plt.scatter(
    pca_plot_frame["PC1"].to_numpy(),
    pca_plot_frame["PC2"].to_numpy(),
    c=pca_plot_frame["cluster_id"].to_numpy(),
    s=18,
    alpha=0.65,
)
plt.title("Segmenty klientów po redukcji PCA do 2 wymiarów")
plt.xlabel(
    f"PC1 ({pca_2d.explained_variance_ratio_[0]:.1%} wariancji)"
)
plt.ylabel(
    f"PC2 ({pca_2d.explained_variance_ratio_[1]:.1%} wariancji)"
)
legend = plt.legend(
    *scatter.legend_elements(),
    title="cluster_id",
    loc="best",
)
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()



### 30. Jak powstają komponenty?

Każdy komponent jest kombinacją liniową wszystkich cech wejściowych. Współczynniki nazywamy często `loadings`.

Duża dodatnia lub ujemna wartość oznacza, że dana cecha silniej wpływa na kierunek komponentu. Znak sam w sobie jest umowny — cały komponent można odwrócić bez zmiany informacji.


In [ ]:

pca_loadings = pl.DataFrame(
    {
        "feature": CLUSTER_FEATURES,
        "PC1_loading": pca_2d.components_[0],
        "PC2_loading": pca_2d.components_[1],
    }
).with_columns(
    (
        pl.col("PC1_loading").abs()
        + pl.col("PC2_loading").abs()
    ).alias("combined_absolute_loading")
).sort(
    "combined_absolute_loading",
    descending=True,
)

pca_loadings



### 31. PCA nie jest wyborem kolumn

- **Redukcja wymiarów** tworzy nowe komponenty z wielu cech.
- **Wybór cech** zachowuje wybrane oryginalne kolumny i usuwa pozostałe.

PCA może poprawić wizualizację albo ograniczyć liczbę zmiennych, ale komponenty są zwykle mniej intuicyjne biznesowo niż takie cechy jak `total_revenue` czy `number_of_orders`.



## Podsumowanie części 3 — Online Retail

```text
7 cech klienta
→ StandardScaler
→ 2 komponenty PCA
→ wykres segmentów K-Means
```

Najważniejsza lekcja:

> PCA pozwala zobaczyć strukturę danych wielowymiarowych, ale wykres 2D jest uproszczoną projekcją, a nie pełnym obrazem wszystkich zależności.



---

## Część 4. Nietypowi klienci względem segmentów

K-Means może służyć nie tylko do segmentacji. Dla każdego klienta możemy obliczyć odległość od centroidu klastra, do którego został przypisany.

Klient bardzo oddalony od centrum swojego segmentu może:

- łączyć nietypowe cechy,
- być przypadkiem granicznym między segmentami,
- reprezentować klienta hurtowego lub wyjątkowo aktywnego,
- zawierać błąd albo wymagać ręcznej weryfikacji.

Jest to **heurystyka anomalii względem segmentów**, a nie uniwersalna definicja anomalii.


In [ ]:

distances_to_centroids = kmeans.transform(X_scaled)
assigned_centroid_distance = distances_to_centroids[
    np.arange(len(kmeans_labels)),
    kmeans_labels,
]

CENTROID_ANOMALY_QUANTILE = 0.99
centroid_distance_threshold = float(
    np.quantile(
        assigned_centroid_distance,
        CENTROID_ANOMALY_QUANTILE,
    )
)

customer_segments = customer_segments.with_columns(
    pl.Series(
        "centroid_distance",
        assigned_centroid_distance,
    ),
    pl.Series(
        "is_centroid_anomaly",
        assigned_centroid_distance >= centroid_distance_threshold,
    ),
)

print(
    "Próg odległości:",
    round(centroid_distance_threshold, 3),
)
print(
    "Klienci oznaczeni jako nietypowi:",
    customer_segments["is_centroid_anomaly"].sum(),
)


In [ ]:

customer_segments.filter(
    pl.col("is_centroid_anomaly")
).select(
    "customer_id",
    "cluster_id",
    "centroid_distance",
    "recency_days",
    "number_of_orders",
    "total_revenue",
    "average_order_value",
    "number_of_products",
    "purchase_frequency_30d",
    "return_rate",
    "dbscan_cluster_id",
    "dbscan_is_noise",
).sort(
    "centroid_distance",
    descending=True,
).head(15)



### Porównanie z DBSCAN

- `is_centroid_anomaly=True` oznacza klienta odległego od środka segmentu K-Means.
- `dbscan_is_noise=True` oznacza klienta, który nie znalazł się w odpowiednio gęstym obszarze.

Te dwie metody mogą wskazać różne rekordy, ponieważ wykorzystują różne definicje nietypowości.


In [ ]:

anomaly_overlap = customer_segments.select(
    pl.len().alias("customers"),
    pl.col("is_centroid_anomaly").sum().alias(
        "centroid_anomalies"
    ),
    pl.col("dbscan_is_noise").sum().alias(
        "dbscan_noise"
    ),
    (
        pl.col("is_centroid_anomaly")
        & pl.col("dbscan_is_noise")
    ).sum().alias("flagged_by_both"),
)

anomaly_overlap


In [ ]:

# Ponowny zapis tabeli po dodaniu wyników wykrywania anomalii.
customer_segments.write_parquet(SEGMENTS_PATH)
print(f"Zaktualizowano: {SEGMENTS_PATH}")
